# Classes as Object Factories
Instead of treating **Classes as Object Factories** as a small isolated topic, the goal here is to treat it as part of Python's larger runtime story: objects are created, names are bound, frames execute bytecode, and the interpreter keeps following rules whether or not the source code looks simple from the outside.

If this topic ever felt a little too neat or too magical when explained quickly, that is exactly what this rewrite is trying to fix. We are going to keep asking two grounding questions: what changed in memory, and what instructions is the interpreter actually stepping through under the surface?

The point is not to drown in internals. The point is to make the language feel explainable. Once that happens, even advanced behavior starts feeling much less mysterious.


When people struggle with **Classes as Object Factories**, the struggle is often not about syntax. It is usually about trying to reason from surface appearance alone. Python rewards a deeper habit: do not ask only what the code *looks like*; ask what objects exist, what names or attributes point to them, what bytecode-level actions are likely happening, and which runtime protocol is being used.

That habit is what turns memorized rules into real understanding. It also makes interviews easier, because you stop giving slogan answers and start giving mechanism-based answers.


- Treat classes as runtime objects
- Understand instance creation clearly
- See how methods become bound to instances
- Connect attribute access to the object model


Creating an instance allocates a new object and then usually initializes it through `__init__`. The class object itself already exists and stores behavior that instances can use through attribute lookup.


In [1]:
class User:
    def __init__(self, name):
        self.name = name

u = User("Ada")
print(type(User))
print(type(u))
print(id(User), id(u))


<class 'type'>
<class '__main__.User'>
1923227145424 1923246190736


Disassembling instance creation shows that class calls are runtime operations like any other call. Python loads the class object, calls it, and receives a new instance back.


In [2]:
import dis

def make_user():
    class Temp:
        pass
    return Temp()

dis.dis(make_user)


  3           0 RESUME                   0

  4           2 PUSH_NULL
              4 LOAD_BUILD_CLASS
              6 LOAD_CONST               1 (<code object Temp at 0x000001BFCA6D01D0, file "C:\Users\VIHAAN\AppData\Local\Temp\ipykernel_5044\727344181.py", line 4>)
              8 MAKE_FUNCTION            0
             10 LOAD_CONST               2 ('Temp')
             12 PRECALL                  2
             16 CALL                     2
             26 STORE_FAST               0 (Temp)

  6          28 PUSH_NULL
             30 LOAD_FAST                0 (Temp)
             32 PRECALL                  0
             36 CALL                     0
             46 RETURN_VALUE

Disassembly of <code object Temp at 0x000001BFCA6D01D0, file "C:\Users\VIHAAN\AppData\Local\Temp\ipykernel_5044\727344181.py", line 4>:
  4           0 RESUME                   0
              2 LOAD_NAME                0 (__name__)
              4 STORE_NAME               1 (__module__)
              6 LOA

That is why they can be passed around, stored, and even customized dynamically.

That is usually stored in attributes like `self.name`.

When accessed through an instance, they become bound methods.

The object model is really a lookup and binding story.


One object carries data and exposes methods that use that data.


In [3]:
class User:
    def __init__(self, name):
        self.name = name

    def greet(self):
        return f"Hello, {self.name}"

u = User("Ada")
print(u.name)
print(u.greet())


Ada
Hello, Ada


Writing `User("Ada")` is really calling the class object.


In [4]:
class Box:
    pass

print(callable(Box))
print(Box())


True


The instance is automatically supplied as the first argument when you call through the instance.


In [5]:
class Greeter:
    def hello(self, name):
        return f"Hello, {name}"

g = Greeter()
print(Greeter.hello)
print(g.hello)
print(g.hello("Python"))


<function Greeter.hello at 0x000001BFCA67B600>
<bound method Greeter.hello of <__main__.Greeter object at 0x000001BFCA6BF310>>
Hello, Python


This is not only a classroom topic. **Classes as Object Factories** usually appears in real projects as a debugging problem, a design choice, a performance tradeoff, or a readability decision. In other words, this topic matters most when the codebase gets large enough that vague intuition stops being reliable.

That is why the low-level view is useful. It gives you something firmer than guesswork when code starts behaving in surprising ways.


- Thinking `self` is a keyword rather than a conventional parameter name
- Using classes for everything even when a simpler structure would do
- Ignoring the fact that classes themselves are objects


- What is a class in Python?
- What happens when you instantiate a class?
- Why does a method receive `self`?


- Classes are callable objects.
- Instances hold state.
- Methods are bound through attribute access.


If this notebook did its job, **Classes as Object Factories** should now feel less like a bag of special rules and more like a natural consequence of Python's runtime model. That is the standard we want: not just recall, but a calm explanation of what the interpreter is seeing and why the behavior follows from that.


A better way to understand Classes as Object Factories is to keep moving back and forth between three views of the same code.

The first view is the human view. This is what the code is trying to say. The second view is the object view. In that view, we ask which Python objects exist, which names refer to them, and which objects are being created, reused, mutated, or discarded. The third view is the interpreter view. In that view, we ask what protocol is being used, what bytecode is being executed, and which runtime rules are controlling the result.

Many learners stop after the first view. That is enough to copy code, but it is usually not enough to explain surprising behavior. Deep understanding starts when these three views become one mental movement instead of three unrelated topics.


There is also a fourth habit that helps a lot: failure-based thinking.

For Classes as Object Factories, ask yourself what normally goes wrong when the idea is only half-understood. Does someone confuse identity with equality? Do they expect copying where only rebinding happened? Do they expect a fresh iterator when they really have an exhausted one? Do they trust a default value that is secretly shared? Do they think a class attribute is per-instance state? Once you can predict the common failure modes before they happen, your understanding becomes much more stable.


In [6]:
import dis
import inspect

def probe_runtime(value):
    local_copy = value
    frame = inspect.currentframe()
    print('locals now:', frame.f_locals)
    return local_copy

print(probe_runtime('demo'))
print('\nbytecode for probe_runtime:')
dis.dis(probe_runtime)


locals now: {'value': 'demo', 'local_copy': 'demo', 'frame': <frame at 0x000001BFCA4F7E80, file 'C:\\Users\\VIHAAN\\AppData\\Local\\Temp\\ipykernel_5044\\939914315.py', line 7, code probe_runtime>}
demo

bytecode for probe_runtime:
  4           0 RESUME                   0

  5           2 LOAD_FAST                0 (value)
              4 STORE_FAST               1 (local_copy)

  6           6 LOAD_GLOBAL              0 (inspect)
             18 LOAD_METHOD              1 (currentframe)
             40 PRECALL                  0
             44 CALL                     0
             54 STORE_FAST               2 (frame)

  7          56 LOAD_GLOBAL              5 (NULL + print)
             68 LOAD_CONST               1 ('locals now:')
             70 LOAD_FAST                2 (frame)
             72 LOAD_ATTR                3 (f_locals)
             82 PRECALL                  2
             86 CALL                     2
             96 POP_TOP

  8          98 LOAD_FAST         

The deeper object-model gain here is to inspect instance namespaces, class namespaces, method resolution order, and descriptor objects directly. That shifts the topic from ?class syntax? to ?runtime lookup process?, which is where the real explanatory power lives.


In [7]:
class Base:
    kind = 'base'
    def hello(self):
        return 'hello'

class Child(Base):
    pass

obj = Child()
print('instance dict:', obj.__dict__)
print('class dict sample:', list(Child.__dict__.keys())[:8])
print('mro:', Child.__mro__)
print('bound method:', obj.hello)


instance dict: {}
class dict sample: ['__module__', '__doc__']
mro: (<class '__main__.Child'>, <class '__main__.Base'>, <class 'object'>)
bound method: <bound method Base.hello of <__main__.Child object at 0x000001BFCA6C08D0>>


The deeper idea here is not simply ?classes create objects?. It is that classes are themselves objects participating in the same runtime world as everything else. They have namespaces, identities, attributes, and types of their own. Once that clicks, object-oriented Python feels less like a separate paradigm and more like another extension of the same object model you have already been studying.


## Final Takeaway

The deepest way to revise **Classes as Object Factories** is to stop treating it as a collection of syntax facts and instead treat it as a runtime story. Ask what objects are involved, what names or attributes point to those objects, what frame is active, and what protocol or bytecode path Python is following. If you can explain this topic at those four levels, then you understand much more than the visible syntax.

In interview terms, the strongest answer is usually not the shortest definition. It is a calm explanation of what Python is doing under the surface and why the behavior follows from that model. For revision, come back to this notebook and ask yourself: could I explain this entire chapter with one small code example, one memory-level explanation, and one interpreter-level explanation? If yes, the topic is becoming yours.
